# Acehnese → Indonesian: train our model once
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**.
3. The first cell asks for your **Hugging Face token** (huggingface.co/settings/tokens, type *Write*). Paste it, press **Enter**.
   It is only used to store your trained model privately in your account so your Mac can download it.
4. Wait about 40–60 minutes with this tab open. You get the results table, and your model is saved.

Then on your Mac: `cd ~/aceh_code && git pull && ./run_app.sh` opens the app at http://127.0.0.1:7860.
It downloads your model once; after that it starts in seconds, offline. No need to train again.
If something goes wrong, copy the red error text and send it to Claude.

In [1]:
# ---- settings ----------------------------------------------------------------------------------
from getpass import getpass
HF_TOKEN = getpass('Paste your Hugging Face WRITE token, then press Enter (Enter alone = download the model as a zip instead): ').strip()
DOWNLOAD_ZIP = False   # True: also download the trained model to this computer as a zip (about 1.2 GB)
USE_QURAN = True       # Acehnese Quran translation (Tgk. Mahjiddin Jusuf): Andrie agreed; False leaves it out
USE_SEED_PIVOT = True  # NLLB-Seed: Acehnese sentences (human) + Indonesian made by NLLB from the English (machine). Run once with True and once with False, compare the scores
EPOCHS = 10            # maximum; training stops earlier when it stops improving

Paste your Hugging Face WRITE token, then press Enter (Enter alone = download the model as a zip instead): ··········


In [2]:
# ---- code + packages + GPU check ---------------------------------------------------------------
import os
REPO = os.path.abspath('aceh_code')
if not os.path.isdir(REPO):
    !git clone -q -b claude/dataset-product-build-108xu8 https://github.com/Mutaib-yye/aceh_code {REPO}
%cd {REPO}
!git pull -q
!pip install -q "transformers==5.19.0" "sentencepiece>=0.2" "gradio==6.29.1" pymupdf pysbd datasketch rapidfuzz sacrebleu
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Menu: Runtime > Change runtime type > T4 GPU > Save. Then Runtime > Run all again.')
print('GPU:', torch.cuda.get_device_name(0))

/content/aceh_code
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 13.1 MB/s eta 0:00:00
GPU: Tesla T4


In [3]:
# ---- training data: NusaX + FLORES-200 + Hikayat Abu Sammah + Wikipedia + Hikayat Malik Mawot -> data/final/combined (leak-checked) ----
!python -m acehid fetch-nusax
if not os.path.exists('data/final/flores/dev.json'):
    !wget -q -O /tmp/flores200_dataset.tar.gz https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && python -m acehid flores /tmp/flores200_dataset.tar.gz || echo 'FLORES download failed: training without it (still fine)'
extra = '--quran' if USE_QURAN else ''
if USE_QURAN:
    !python -m acehid fetch-quran
extras = ['data/final/wikipedia/pairs.json']                      # Wikipedia ace-id pairs (CC-BY-SA), already in the repo
if os.path.exists('SKRIPSI FULL INAYATI.pdf'):                    # Hikayat Malik Mawot: verse + translation, built from the thesis PDF
    !python -m acehid malik-mawot "SKRIPSI FULL INAYATI.pdf"
    extras.append('data/human/malik_mawot_pairs.json')
else:
    print('thesis PDF not found: training without Hikayat Malik Mawot')
if USE_SEED_PIVOT:                                                # about 5-10 minutes on the T4 GPU
    !python -m acehid seed-pivot
    extras.append('data/final/seed_pivot/pairs.json')
!python -m acehid build-train {extra} --extra {' '.join(extras)}
import json
m = json.load(open('data/final/combined/manifest.json'))
print('training pairs:', m['train']['n'], m['train']['by_source'])
print('validation:', m['validation']['n'], '| test sets:', {k: v['n'] for k, v in m['tests'].items()})

downloaded to /content/aceh_code/data/raw/external/nusax
[flores] dev=997 devtest=1012
{"dev": 997, "devtest": 1012}
downloaded to /content/aceh_code/data/raw/external/quran
[malik-mawot] segments=1030 pairs=350
{"segments": 1030, "pairs": 350, "out": "/content/aceh_code/data/human/malik_mawot_pairs.json"}
config.json: 100% 846/846 [00:00<00:00, 2.86MB/s]
tokenizer_config.json: 100% 564/564 [00:00<00:00, 1.90MB/s]

sentencepiece.bpe.model: downloading bytes:   0% 0.00/4.85M [00:00<?, ?B/s]
sentencepiece.bpe.model: downloading bytes: 100% 3.48M/3.48M [00:00<00:00, 5.77MB/s,  344kB/s  ]
sentencepiece.bpe.model: reconstructing file: 100% 4.85M/4.85M [00:00<00:00, 8.04MB/s,  479kB/s  ]

tokenizer.json: downloading bytes:  23% 3.96M/17.3M [00:00<00:01, 8.99MB/s]
tokenizer.json: downloading bytes: 100% 7.83M/7.83M [00:00<00:00, 14.0MB/s,  769kB/s  ]
tokenizer.json: reconstructing file: 100% 17.3M/17.3M [00:00<00:00, 31.0MB/s, 1.71MB/s  ]
special_tokens_map.json: 100% 3.55k/3.55k [00:00<00:00

In [4]:
# ---- fine-tune NLLB-200 (about 30-90 minutes on a T4, longer with more data; not measured) ----------------------------------------------
!python -m acehid train --epochs {EPOCHS}
if not os.path.exists('models/ace-id-nllb/RESULTS.md'):
    print('First attempt did not finish (usually: not enough GPU memory). Retrying with a smaller batch...')
    !python -m acehid train --epochs {EPOCHS} --batch-size 4
TRAINED = os.path.exists('models/ace-id-nllb/RESULTS.md')
print('TRAINING FINISHED' if TRAINED else 'TRAINING FAILED: copy the red error above and send it to Claude. The app will be published with the base model.')

device=cuda train_pairs=14653 dev=379 tests={'nusax': 400, 'flores': 1012, 'hikayat': 143} base=facebook/nllb-200-distilled-600M
Loading weights: 100% 512/512 [00:00<00:00, 14657.99it/s]
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Bot

In [5]:
# ---- results: base model vs ours on test sentences it never saw ----------------------------------
from IPython.display import Markdown, display
if TRAINED:
    display(Markdown(open('models/ace-id-nllb/RESULTS.md', encoding='utf-8').read()))
else:
    print('No results table: training did not finish.')

# Fine-tuning results: facebook/nllb-200-distilled-600M -> models/ace-id-nllb

Training pairs: 14653 (flores 797, hikayat 1432, malik_mawot_pairs 350, nusax 500, pairs 6108, quran 5466); best epoch 6 of 8 (early stopping on validation chrF++).
Test sets were never trained on and never used to pick the epoch. Higher is better.

| test set | n | base chrF++ | fine-tuned chrF++ | change | base BLEU | fine-tuned BLEU |
|---|---|---|---|---|---|---|
| nusax | 400 | 38.66 | 56.6 | +17.94 | 13.53 | 30.2 |
| flores | 1012 | 46.27 | 51.97 | +5.70 | 21.31 | 25.71 |
| hikayat | 143 | 23.98 | 60.48 | +36.50 | 2.68 | 36.04 |

## Examples: nusax

| Acehnese | reference | base | fine-tuned |
|---|---|---|---|
| Toe ngon hotel lon dom, jak ngon tapak, disinoe le that pilehan makananjih, teumpat nyang luwah, bereh | Dekat dengan hotel saya menginap, hanya ditempuh jalan kaki, di sini banyak sekali pilihan makanannya, tempat yang luas, dan menyenangkan | Di dekat hotel saya, berjalan kaki, ada banyak pilihan makanan, tempat yang luas, tempat beristirahat | Dekat dengan hotel saya menginap, berjalan kaki, di sini banyak pilihan makanannya, tempat yang luas, lumayan |
| Nyo beutoi, jih teungoh jaga warong. | Iya benar, dia sedang jaga warung. | Ini benar, karena dia sedang menjaga warganya. | Benar, dia sedang menjaga warung. |
| On rumpunjih lumayan tapi bieng saos padang cukop peuget kecewa, kamoe dijok bieng niet asoe, keuneulheueh hana kamoe pajoh bieng nyan dan kamoe jok pulang teuma | Kangkungnya lumayan tapi kepiting saus padangnya mengecewakan kami dikasih kepiting yang kopong akhir kami tidak makan keptingnya dan dikembalikan. | Dan kami tidak makan dagingnya, dan kami tidak makan dagingnya, lalu kami pulang ke rumah kami. | On rumpunya lumayan tapi biji saos padang cukup mengecewakan, kami diberi biji niat daging, setelah itu kami tidak makan dan kami mengulanginya pulang |
| Meuteumpat di braga city walk nyang saboh geudong deungon aston ngon fave hotel, teumpat nyoe cukuop nyaman keu duek-duek jep kupi. Kupi meujampu ngon te nyang phon that lon rasa rupajih mangat, peu lom ditamah ngoen boh manoh seutengoh matang jeuet keu pengon sarat duek khem ngoen rakan-rakan. Area nyang hana jeuet piep rukok meutamah hayeue sara ta nikmati pemandangan ureueng teubit tamong mal nyoe. | Bertempat di braga city walk yang satu gedung dengan aston dan fave hotel, tempat ini sangat nyaman buat kongkow-kongkow. Kopi campur teh yang baru pertama kali saya nikmati ternyata sangat enak, dipadu dengan telur setengah matang menjadi pendamping mengobrol bersama teman-teman. Area yang bebas merokok semakin mengasyikkan sambil menikmati pemandangan lalu lalang orang-orang yang keluar masuk mal ini. | Tempat ini terletak di tengah jalan kota yang dibangun dengan aston dan fave hotel, tempat ini cukup nyaman untuk duduk di atasnya. Di antara minuman yang pertama kali saya rasakan, ditambah buah-buahan yang sudah matang menjadi halangan untuk berkemah dengan teman-teman. | Terletak di braga city walk yang satu bangunan dengan aston dan fave hotel, tempat ini cukup nyaman buat nongkrong-nongkrong buat minum kopi. Kopi dicampur dengan teh pertama kali saya merasa rasanya enak, apalagi tambahan dan menu semi matang yang bisa membuat betah untuk nongkrong bersama teman teman. Daerah yang tidak dapat dimasuki semakin menarik untuk menikmati pemandangan pengunjung yang tiba di mal ini. |
| Gianjar teurimong bantuan sosial dua ribe lapan blah (2018) sedupna peut ploh peut, sikureng (44,9) miliar | Gianyar terima bantuan sosial 2018 sebesar rp 44, 9 miliar | Gianjar menerima bantuan sosial dua ratus delapan belas (2018) sementara empat puluh empat, senilai (44,9) miliar | Gianjar menerima bantuan sosial sebanyak dua ribu delapan belas (2018) dan sebanyak empat puluh empat, sembilan (44,9) miliar. |
| Nyoe keuh salah saboh geuneurasi nyang hana guna. Daripada beumanfaat keu bangsa, jipileh jeut keu bahaya keu bangsa! Mantap tuntut laju ngon UU ITE | Ini salah satu contoh generasi yang tidak berguna. Bukan malah bermanfaat untuk bangsa tapi malah jadi sesuatu yang berbahaya bagi bangsa! Mantap jerat saja pakai uu ite | Ini adalah satu-satunya kesombongan yang tidak berguna dan tidak berguna bagi bangsa, dan bukan untuk kepentingan bangsa. Perhatikanlah hukum-hukum ini. | Ini adalah salah satu generasi yang tidak berguna. Daripada memberi manfaat kepada bangsa, dipilih jadi bahaya bagi bangsa! Mantap terus dengan undang-undang ITE |
| Hai rakan, jinoe teungoh jipeupah jadi aplikasi hanjeuet diakses. | Sore sahabat, saat ini sedang dilakukan pemeliharaan sehingga aplikasi tidak dapat diakses. | Teman-teman, aplikasi ini sedang disewa sehingga tidak dapat diakses. | Wahai teman, sekarang sedang dibolak-balikkan jadi aplikasi tidak dapat diakses. |
| Cit leumoh tanyoe nyoe loen pih hana beuhe sebeutoi jih | Memang lemah kita ini aku juga penakut sebenarnya | Kami lemah, dan kami tidak kuat untuk melawan Dia. | Maka sungguh, kami (juga) tidak berdaya dan tidak berdaya |

## Examples: flores

| Acehnese | reference | base | fine-tuned |
|---|---|---|---|
| "Kamoe jinoe na tikoih umu 4 buleuen nyang hana diabetes nyang awaijih saket diabetes," geutamah le gobnyan. | “Saat ini ada mencit umur 4 bulan nondiabetes yang dulunya diabetes,” tambahnya. | "Kami sekarang memiliki anak usia 4 bulan yang tidak memiliki diabetes yang awalnya menderita diabetes", tambahnya. | "Kami sekarang memiliki tikus berusia 4 bulan tanpa diabetes yang awalnya menderita diabetes," tambah dia." |
| Dr. Ehud Ur, profesor kedokteran bak Universitas Dalhousie di Halifax, Nova Scotia ngôn keupala divisi klinis ngôn ilmiah dari Asosiasi Diabetes Kanada geupeuingat bahwa penelitian nyoe mantong lam masa nyang keuphôn. | Dr. Ehud Ur, profesor kedokteran di Dalhousie University di Halifax, Nova Scotia dan ketua divisi klinis dan ilmiah di Perhimpunan Diabetes Kanada memperingatkan bahwa penelitiannya masih berada di tahap awal. | Dr. Ehud Ur, profesor kedokteran di Universitas Dalhousie di Halifax, Nova Scotia, kepala divisi klinis linguistik ilmiah dari Asosiasi Diabetes Kanada memperingatkan bahwa penelitian ini masih dalam tahap awal. | Dr. Ehud Ur, profesor kedokteran di Universitas Dahlhousie di Halıfaks, Nova Scotia dan kepala divisi klinis dan ilmiah dari Asosiasi Diabet Kanada, memperingatkan bahwa penelitian ini masih dalam fase awal. |
| Lagèe ladom ureueng carong laenjih, gobnyan ragu peukeuh peunyaket diabetes jeut puléh, sabab nyang jiteumèe nyoe hana meusangkot deungôn ureueng nyang ka meupeunyaket diabetes jeunèh 1. | Seperti halnya ahli-ahli lain, dia merasa skeptis apakah diabetes dapat disembuhkan, menegaskan bahwa penemuan-penemuan ini tidak relevan dengan orang-orang yang telah mengidap diabetes Jenis 1. | Seperti orang lain, dia meragukan apakah penyakit diabetes bisa sembuh, karena penemuan ini tidak melibatkan semua orang yang telah menderita diabetes seri 1. | Seperti halnya banyak ilmuwan lainnya, mereka meragukan apakah penyakit diabetes bisa sembuh, karena temuan mereka tidak terkait dengan penderita diabetes tipe 1. |
| Bak uroe Senin, Sara Danius, sekretaris teutap Komite Nobel Sastra bak Akademi Swedia, geuumumkan ngôn bak mandum ureueng lam masa program radio di Radio Sveriges di Swedia komite nyan, han jeuet meusambông deungôn Bob Dylan ngôn lansông teuntang keumenangan Hadiah Nobel 2016 lam Sastra, ka geupeutinggai usaha keu meusambông deungôn gobnyan. | Sara Danius, sekretaris permanen Komite Nobel Literatur di Akademi Swedia, pada hari Senin mengumumkan kepada publik selama berlangsungnya program radio di Sveriges Radio di Swedia bahwa komite tersebut tidak bisa menghubungi Bob Dylan secara langsung untuk menyampaikan kemenangannya atas Hadiah Nobel 2016 di bidang Literatur. Sara akhirnya menyerah menghubunginya. | Pada hari Senin, Sara Danius, sekretaris permanen Komite Nobel Sastra di Swedia Academy, mengumumkan kepada semua orang selama program radio di Radio Sveriges di Swedia bahwa komite itu tidak akan bertemu dengan Bob Dylan dalam pidato tentang pemenang Hadiah Nobel Sastra 2016, telah meninggalkan upaya untuk bertemu dengannya. | Sara Danius, sekretaris tetap Komite Nobel Sastra di Swedia Academy, pada hari Senin mengumumkan kepada seluruh staf siaran radio Swedia bahwa ia tidak akan dapat terhubung dengan Bob Dylan dan langsung mengumumkan bahwa pemenang Hadiah Nobel Sastra 2016 telah meninggalkan upaya untuk terhubung dengannya. |
| Danius kheun, "Jinoe kamoe hana meubuet sapeue. Lôn ka lôn talipun ngon lôn kirém email keu kolaborator gobnyan nyan paléng rab ngôn lôn teurimong balah nyang paléng ruméh. Jinoe, nyan ka sep. | "Danius berkata, "Saat ini kita tidak melakukan apa-apa. Saya sudah menelepon dan mengirimkan surel kepada kolaborator terdekatnya dan memperoleh balasan yang sangat bersahabat. Untuk saat ini hal tersebut sudah cukup." | Dan Daniel menjawab, "Tidak ada yang bisa kami lakukan. Saya telah menelepon dan mengirim email kepada rekan-rekannya, dan saya akan menerima bagian yang paling besar. Sekarang sudah selesai. | Danius berkata, "Saat ini kami tidak melakukan apa pun. Saya telah menelepon dan mengirim email kepada kolaborator mereka yang paling dekat dan saya akan menerima balasannya yang paling ramah. Untuk saat ini, sudah cukup. " |
| Sigohlom gobnyan, CEO Ring, Jamie Siminoff, geukheun keu perusahaan phôn watèe bel pinto gobnyan han jeuet meudeungo dari keudèe gobnyan lam garasi gobnyan. | Sebelumnya, CEO Ring, Jamie Siminoff, berkomentar bahwa perusahaan tersebut bermula saat bel rumahnya tidak terdengar dari tokonya yang berada di garasi. | Sebelumnya, CEO Ring, Jamie Siminoff, mengatakan kepada perusahaan pembuat bel bahwa dia tidak bisa mendengar suara dari orang-orang di garasinya. | Sebelumnya, CEO Ring, Jamie Siminoff, mengatakan kepada perusahaan pertama bahwa lonceng tidak dapat didengar dari istrinya di garasi. |
| Gobnyan peugot bel pinto WiFi, jikheun lé gobnyan. | Ia melakukan pembuatan bel pintu dengan teknologi WiFi, katanya. | Dia membuat bel pintu WiFi, katanya. | Dia membuat bel pintu wifi, katanya. |
| Siminoff kheun hasé meukat geuék 'oh lheuh piasan gobnyan bak thôn 2013 lam episode Shark Tank lam pat panel acara tula bri dana keu startup. | Siminoff mengatakan penjualan meningkat setelah ia muncul di episode Shark Tank 2013, ketika panel acara tersebut menolak membiayai perusahaan rintisan. | Siminoff mengatakan hasil perdagangan meningkat setelah perannya pada tahun 2013 dalam episode Shark Tank di mana panel acara memberi dana untuk startup. | Siminoff mengatakan penjualan meningkat setelah pertunjukan tahunnya pada tahun 2013 dalam sebuah episode dari "Shark tank" di mana panel tuan rumah memberikan dana untuk perusahaan awal. |

## Examples: hikayat

| Acehnese | reference | base | fine-tuned |
|---|---|---|---|
| Sabda Uma beurtitah le, wahe aneuk jantong hate | Sabda Umar bertitah lagi, wahai anak jantung hati | Kata-kata yang diucapkan oleh orang banyak, kata-kata yang diucapkan oleh anak-anak yang dicintai. | Sabda Umar bertitahlah, wahai anak jantung hati |
| Bek tamoe gundah lon pike, menggoh hase ban had gata | Jangan menangis sudah kupikir, sebelum selesai pada had anda | Aku akan menangis karena takut, Aku akan menangis karena takut, Aku akan menangis karena takut. | Jangan menangis gundah kupikir, menangguhkan hasil bagai had anda |
| saba ancuk nyang meutuwah, rila ateueh hukom Allah | Sabarlah nak bertuah, relalah atas hukum Allah | Dengan kesabaran dan kesabaran yang besar, orang-orang yang taat pada hukum Allah | Sabar anak yang bertuah, rela pada hukum Allah |
| Dilee nabiyullah, bala susah nibak gata | Dahulu Nabiyullah, lebih susah dari pada anda | Hai nabi-nabi, cilakalah engkau! | Dahulu Nabiyullah, bala susah dari anda |
| Padum-padum Nabi Adam, syereuga neuweh di makam | Betapakah Nabi Adam, syurga pindah di makam | Ada orang-orang yang mati, ada orang-orang yang mati, ada orang-orang yang mati, ada orang-orang yang mati, ada orang-orang yang mati, ada orang-orang yang mati, ada orang-orang yang mati, ada orang-orang yang mati, ada orang-orang mati, ada orang-orang mati, ada orang-orang mati, ada orang-orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati, ada orang mati | Beberapa Nabi Adam, surga dipenuhi di makam |
| Boh khuldi seubab deundam, seru alam po meureuka | Buah khuldi sebab dendam, seru alam Tuhan murka | Orang-orang berdarah berteriak-teriak karena kehancuran, dan orang-orang berdarah berteriak-teriak karena kemarahan. | Baiklah khuldi sebab demam, seru alam tuan murka |
| padum seksa Nabi Nuh, dalam la-ot yoh ie damoh | Betapa siksa Nabi Nuh, dalam laut saat air bah | Pada hari-hari pertama, ketika Nuh berada di dalam laut, | Berapa siksa Nabi Nuh, ke dalam laut kala air berhembus |
| Dum karam han leumah tanoh, eneuk jroeh leubeh geusaba | Semua karam tak tampak tanah, anak indah lebih sabarnya | Tanah yang kering tidak akan terlihat lagi, dan air yang kering tidak akan terlihat lagi. | Semua karam tak tampak tanah, nanti cuaca lebih bersabar |


In [6]:
# ---- the professor's items: direct, transfer and neural side by side + dictionary panel (disambiguation) ------------
# Downloads Andrie's dictionary list (kamus_aceh_indonesia_1985.csv) from the project's GitHub repo; no upload needed.
import os, json, subprocess, sys, importlib.util
spec = importlib.util.spec_from_file_location('space_core', 'space/core.py'); _core = importlib.util.module_from_spec(spec); spec.loader.exec_module(_core)
GLOSS = _core.ensure_glossary()
if GLOSS:
    subprocess.run([sys.executable, '-m', 'acehid', 'compare-approaches'], check=True, stdout=subprocess.DEVNULL)
    rules = json.load(open('reports/mt_approaches.json'))
    nusax = json.load(open('models/ace-id-nllb/training_log.json'))['tests'].get('nusax')
    print('NusaX test, %d sentences (chrF++ / BLEU, higher is better)' % rules['_n'])
    for name, v in rules.items():
        if not name.startswith('_'):
            print('  %-40s %5.1f / %4.1f' % (name, v['chrF++'], v['BLEU']))
    if nusax:
        print('  %-40s %5.1f / %4.1f' % ('neural: base NLLB-200', nusax['base']['chrF++'], nusax['base']['BLEU']))
        print('  %-40s %5.1f / %4.1f' % ('neural: our fine-tuned model', nusax['finetuned']['chrF++'], nusax['finetuned']['BLEU']))
    print('Data augmentation: NLLB-Seed pivot is', 'ON' if USE_SEED_PIVOT else 'OFF', '(see the training pair counts above).')
    print('Disambiguation: the app below now has a "Kata per kata" panel that marks the best-fitting meaning with a tick.')
else:
    print('Dictionary list could not be downloaded: skipped. The neural results above are not affected.')


NusaX test, 400 sentences (chrF++ / BLEU, higher is better)
  copy (Acehnese text unchanged)            32.0 /  6.3
  direct (dictionary, word for word)        35.5 /  6.1
  transfer (rules + dictionary)             39.2 /  9.1
  neural: base NLLB-200                     38.7 / 13.5
  neural: our fine-tuned model              56.6 / 30.2
Data augmentation: NLLB-Seed pivot is ON (see the training pair counts above).
Disambiguation: the app below now has a "Kata per kata" panel that marks the best-fitting meaning with a tick.


In [7]:
# ---- try it (change the sentences if you like) ----------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from acehid.normalize import normalize
MODEL_DIR = 'models/ace-id-nllb' if TRAINED else 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(MODEL_DIR, src_lang='ace_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to('cuda').eval()
def tr(text):
    x = tok([normalize(text)[0]], return_tensors='pt').to('cuda')
    with torch.no_grad():
        y = model.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids('ind_Latn'), num_beams=4, max_new_tokens=128)
    return tok.batch_decode(y, skip_special_tokens=True)[0]
for s in ['Lon jak u peukan uroe nyoe.', 'Teuma troih le Abu Sammah, budak indah paraih rupa']:
    print(s, '->', tr(s))

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Lon jak u peukan uroe nyoe. -> Aku pergi ke pasar hari ini.
Teuma troih le Abu Sammah, budak indah paraih rupa -> Lalu sampailah Abu Sammah, budak indah paras rupa


In [8]:
# ---- save your trained model so the app on your Mac can use it (no training again later) ---------
import shutil
if not TRAINED:
    print('Training did not finish, so there is no model to save. Send the red error above to Claude.')
else:
    fp16 = 'models/ace-id-nllb-fp16'                    # half precision: 1.2 GB instead of 2.4 GB, same translations
    model.half().save_pretrained(fp16); tok.save_pretrained(fp16)
    for f in ('training_log.json', 'RESULTS.md', 'README.md'):
        shutil.copy(f'models/ace-id-nllb/{f}', fp16)
    if HF_TOKEN:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            model_repo = f"{api.whoami()['name']}/ace-id-nllb"
        except Exception as e:
            raise RuntimeError('Hugging Face did not accept the token. Create a WRITE token at huggingface.co/settings/tokens, run the first cell again, then this cell.') from e
        api.create_repo(model_repo, private=True, exist_ok=True)
        api.update_repo_settings(model_repo, private=True)
        api.upload_folder(repo_id=model_repo, folder_path=fp16, commit_message='Fine-tuned Acehnese -> Indonesian NLLB')
        print('MODEL SAVED (private):', f'https://huggingface.co/{model_repo}')
        print('On your Mac:  cd ~/aceh_code && git pull && ./run_app.sh   (downloads it once, then opens the app)')
    if not HF_TOKEN or DOWNLOAD_ZIP:
        zip_path = shutil.make_archive(os.path.join(REPO, 'ace-id-nllb'), 'zip', fp16)
        print('zip ready:', zip_path, f'({os.path.getsize(zip_path) / 1e9:.1f} GB). Keep it in your Downloads folder; ./run_app.sh finds it there.')
        try:
            from google.colab import files
            files.download(zip_path)
        except ImportError:
            print('Not in Colab: copy the zip from', zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODEL SAVED (private): https://huggingface.co/mutaib3/ace-id-nllb
On your Mac:  cd ~/aceh_code && git pull && ./run_app.sh   (downloads it once, then opens the app)


In [9]:
# ---- make the model zip to share with friends or take to the presentation (1.2 GB) ----------------
import shutil, os
from google.colab import files
z = shutil.make_archive('/content/ace-id-nllb', 'zip', '/content/aceh_code/models', 'ace-id-nllb-fp16')
print('zip:', z, round(os.path.getsize(z) / 1e9, 2), 'GB')
files.download(z)

zip: /content/ace-id-nllb.zip 1.14 GB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# ---- optional: view the app right here (temporary public link while this tab is open) -------------
import sys, importlib.util, gc
del model; gc.collect(); torch.cuda.empty_cache()
sys.path.insert(0, os.path.join(REPO, 'space'))
spec = importlib.util.spec_from_file_location('space_app', os.path.join(REPO, 'space/app.py'))
app = importlib.util.module_from_spec(spec); spec.loader.exec_module(app)
app.start(MODEL_DIR, share=True)      # prints: Running on public URL: https://xxxx.gradio.live

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5946856d1d5867cc23.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
